# Model Evaluation

In this notebook, we will evaluate several machine learning algorithms on our dataset. The algorithms include Decision Tree, Random Forest, XGBoost, SVM, and Naive Bayes.

## 1. Libraries & imports

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, precision_score, recall_score

## 2. Data Loading and Preprocessing

In [4]:
# Load the dataset
data = pd.read_csv("dataset.csv")

# Splitting the dataset into training and testing sets
X = data.drop(columns=["File", "Cell", "LC", "LEC", "CCC", "LE", "CIC"])
y = data[["LC", "LEC", "CCC", "LE", "CIC"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 3. Training

In [5]:
def evaluate_model(model, name):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    y_true_flat = y_test.values.flatten()
    y_pred_flat = predictions.flatten()
    asc, mcc, f1, gmean, precision, recall = calculate_classification_metrics(y_true_flat, y_pred_flat)
    return [name, asc, mcc, f1, gmean, precision, recall]

## 3. Evaluation

In [6]:
def calculate_classification_metrics(y_true, y_pred):
    asc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    gmean = (precision * recall) ** 0.5
    return asc, mcc, f1, gmean, precision, recall

In [7]:
models = [
    (DecisionTreeClassifier(max_depth=20, min_samples_leaf=1, min_samples_split=5), "DecisionTree"),
    (RandomForestClassifier(max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=50), "RandomForest"),
    (xgb.XGBClassifier(colsample_bytree=0.9, learning_rate=0.1, max_depth=3, n_estimators=50, subsample=0.8), "XGBoost"),
    (OneVsRestClassifier(SVC(C=0.1, gamma='scale', kernel='linear')), "SVM"),
    (OneVsRestClassifier(GaussianNB(var_smoothing=5.3366992312063123e-05)), "GaussianNB")
]

In [8]:
results = []
for model, name in models:
    results.append(evaluate_model(model, name))

# Display results in a table
df_results = pd.DataFrame(results, columns=["Model", "ASC", "MCC", "F1", "Gmean", "Precision", "Recall"])
print(df_results)

          Model       ASC       MCC        F1     Gmean  Precision    Recall
0  DecisionTree  0.965217  0.819243  0.833333  0.837532   0.925926  0.757576
1  RandomForest  0.966957  0.828308  0.840336  0.845396   0.943396  0.757576
2       XGBoost  0.982609  0.914419  0.924242  0.924242   0.924242  0.924242
3           SVM  0.968696  0.838608  0.852459  0.855337   0.928571  0.787879
4    GaussianNB  0.923478  0.613624  0.656250  0.656571   0.677419  0.636364
